# Ayudantía 12: Laboratorio 6 Parte 1 #
Juan Pablo Tapia | jp.tapia@uc.cl

12 de Junio, 2025.

## Importar librerias

In [1]:
!pip install dash dash-bootstrap-components requests

In [2]:
!pip install python-dotenv

In [3]:
from dotenv import load_dotenv
import requests
import pandas as pd
import os
from getpass import getpass
from IPython.display import display
from dash import Dash, html, dcc, Input, Output, State
import dash_bootstrap_components as dbc
import plotly.express as px

## Extracción de datos climáticos y ambientales

Cargar datos.

In [4]:
df_countries = pd.read_csv("countries.csv")
df_cities = pd.read_csv("cities.csv")
df_weather = pd.read_parquet("daily_weather_clean.parquet")
print("Paises:")
display(df_countries.head())
print("\nCiudades:")
display(df_cities.head())
print("\nClima:")
display(df_weather.head())

Paises:


,country,native_name,iso2,iso3,population,area,capital,capital_lat,capital_lng,region,continent
0,Afghanistan,افغانستان,AF,AFG,26023100.0,652230.0,Kabul,34.526011,69.177684,Southern and Central Asia,Asia
1,Albania,Shqipëria,AL,ALB,2895947.0,28748.0,Tirana,41.326873,19.818791,Southern Europe,Europe
2,Algeria,الجزائر,DZ,DZA,38700000.0,2381741.0,Algiers,36.775361,3.060188,Northern Africa,Africa
3,American Samoa,American Samoa,AS,ASM,55519.0,199.0,Pago Pago,-14.275479,-170.704830,Polynesia,Oceania
4,Angola,Angola,AO,AGO,24383301.0,1246700.0,Luanda,-8.827270,13.243951,Central Africa,Africa



Ciudades:


,station_id,city_name,country,state,iso2,iso3,latitude,longitude
0,41515,Asadabad,Afghanistan,Kunar,AF,AFG,34.866000,71.150005
1,38954,Fayzabad,Afghanistan,Badakhshan,AF,AFG,37.129761,70.579247
2,41560,Jalalabad,Afghanistan,Nangarhar,AF,AFG,34.441527,70.436103
3,38947,Kunduz,Afghanistan,Kunduz,AF,AFG,36.727951,68.872530
4,38987,Qala i Naw,Afghanistan,Badghis,AF,AFG,34.983000,63.133300



Clima:


,station_id,city_name,date,season,avg_temp_c,min_temp_c,max_temp_c,precipitation_mm,snow_depth_mm,avg_wind_dir_deg,avg_wind_speed_kmh,peak_wind_gust_kmh,avg_sea_level_pres_hpa,sunshine_total_min
2991,91765,Pago Pago,1973-01-13,Summer,28.3,24.4,31.1,0.0,0.0,117.0,9.5,31.3,1009.5,720.0
3028,91765,Pago Pago,1973-02-19,Summer,29.5,27.2,32.8,4.3,0.0,341.0,18.9,55.4,1006.6,636.0
3039,91765,Pago Pago,1973-03-02,Autumn,28.0,25.0,31.7,0.5,0.0,281.0,14.6,35.3,1008.0,648.0
3063,91765,Pago Pago,1973-03-26,Autumn,28.0,24.4,31.1,4.1,0.0,34.0,11.3,37.1,1010.9,192.0
3131,91765,Pago Pago,1973-06-02,Winter,28.2,25.6,30.6,0.0,0.0,100.0,21.4,37.1,1011.3,354.0


Cargar API KEY.

In [5]:
load_dotenv()
API_KEY = os.getenv("API_KEY")

### a) Pronóstico del clima

Crear función para obtener el pronóstico.

In [6]:
def get_forecast(ciudades, df_cities, api_key):
    df_forecast = pd.DataFrame()
    for ciudad in ciudades:
        ciudad_encontrada = df_cities[df_cities["city_name"] == ciudad]
        if ciudad_encontrada.empty:
            continue
        lat = ciudad_encontrada["latitude"].values[0]
        lon = ciudad_encontrada["longitude"].values[0]
        pais = ciudad_encontrada["country"].values[0]
        url = f"http://api.openweathermap.org/data/2.5/forecast?lat={lat}&lon={lon}&appid={api_key}"
        r = requests.get(url)
        if r.status_code != 200:
            continue
        data = r.json()

        rows = []
        for entry in data['list']:
            rows.append({
                'Ciudad': ciudad,
                'Pais': pais,
                'Fecha y hora': entry['dt_txt'],
                'Descripción': entry['weather'][0]['description'],
                'Temperatura (°C)': entry['main']['temp'] - 273.15,
                'Humedad (%)': entry['main']['humidity'],
                'Sensación térmica (°C)': entry['main']['feels_like'] - 273.15,
                'Velocidad del viento (m/s)': entry['wind']['speed'],
                'Presión atmosférica nivel del suelo (hPa)': entry['main'].get('grnd_level', None),
                'Presión atmosférica nivel del mar (hPa)': entry['main'].get('sea_level', None),
            })
        df = pd.DataFrame(rows)
        df_forecast = pd.concat([df_forecast, df], ignore_index=True)
    promedios = df_forecast.mean(numeric_only=True).to_dict()
    return df_forecast, promedios

Probar funcionamiento con ciudad Santiago.

In [7]:
df_santiago, promedio_santiago = get_forecast(["Santiago"], df_cities, API_KEY)
display(df_santiago)
display(promedio_santiago)

,Ciudad,Pais,Fecha y hora,Descripción,Temperatura (°C),Humedad (%),Sensación térmica (°C),Velocidad del viento (m/s),Presión atmosférica nivel del suelo (hPa),Presión atmosférica nivel del mar (hPa)
0,Santiago,Chile,2025-06-12 18:00:00,moderate rain,8.80,83,7.72,2.09,942,1021
1,Santiago,Chile,2025-06-12 21:00:00,light rain,8.63,82,8.12,1.47,942,1021
2,Santiago,Chile,2025-06-13 00:00:00,light rain,8.85,78,8.37,1.47,942,1021
3,Santiago,Chile,2025-06-13 03:00:00,overcast clouds,8.49,78,8.49,0.97,942,1020
4,Santiago,Chile,2025-06-13 06:00:00,broken clouds,8.04,81,8.04,0.17,942,1021
5,Santiago,Chile,2025-06-13 09:00:00,broken clouds,8.05,81,8.05,0.98,941,1020
6,Santiago,Chile,2025-06-13 12:00:00,scattered clouds,7.37,80,7.37,0.53,943,1022
7,Santiago,Chile,2025-06-13 15:00:00,clear sky,10.19,66,8.99,1.23,943,1021
8,Santiago,Chile,2025-06-13 18:00:00,clear sky,12.12,55,10.82,1.66,942,1020
9,Santiago,Chile,2025-06-13 21:00:00,few clouds,12.24,52,10.88,1.98,942,1019


{'Temperatura (°C)': 10.62250000000002,
 'Humedad (%)': 59.975,
 'Sensación térmica (°C)': 9.722250000000018,
 'Velocidad del viento (m/s)': 1.1465,
 'Presión atmosférica nivel del suelo (hPa)': 942.475,
 'Presión atmosférica nivel del mar (hPa)': 1020.7}

Verificar si el codigo funciona para al menos 100 ciudades.

In [8]:
df_muestra = df_cities.sample(100)
df_muestra.head()

,station_id,city_name,country,state,iso2,iso3,latitude,longitude
610,61272,Ségou,Mali,Ségou,ML,MLI,13.439992,-6.260016
537,40310,Ma'an,Jordan,Ma`an,JO,JOR,30.192003,35.736004
774,84691,Ica,Peru,Ica,PE,PER,-14.067993,-75.725492
74,94970,Hobart,Australia,Tasmania,AU,AUS,-42.850009,147.295030
674,67283,Quelimane,Mozambique,Zambezia,MZ,MOZ,-17.880008,36.889986


In [9]:
ciudades = df_muestra['city_name'].tolist()
df_ciudades, promedio_ciudades = get_forecast(ciudades, df_cities, API_KEY)
display(df_ciudades)
display(promedio_ciudades)

,Ciudad,Pais,Fecha y hora,Descripción,Temperatura (°C),Humedad (%),Sensación térmica (°C),Velocidad del viento (m/s),Presión atmosférica nivel del suelo (hPa),Presión atmosférica nivel del mar (hPa)
0,Ségou,Mali,2025-06-12 18:00:00,few clouds,38.27,24,38.03,5.17,974,1007
1,Ségou,Mali,2025-06-12 21:00:00,few clouds,37.34,27,37.45,3.06,976,1007
2,Ségou,Mali,2025-06-13 00:00:00,scattered clouds,34.83,34,35.31,5.25,978,1010
3,Ségou,Mali,2025-06-13 03:00:00,light rain,28.10,58,29.37,8.81,976,1009
4,Ségou,Mali,2025-06-13 06:00:00,overcast clouds,28.51,44,28.46,2.08,978,1011
...,...,...,...,...,...,...,...,...,...,...
3995,Mahajanga,Madagascar,2025-06-17 03:00:00,overcast clouds,20.24,73,20.23,5.29,1016,1019
3996,Mahajanga,Madagascar,2025-06-17 06:00:00,broken clouds,23.78,60,23.78,5.93,1017,1020
3997,Mahajanga,Madagascar,2025-06-17 09:00:00,broken clouds,28.85,43,28.73,4.76,1015,1018
3998,Mahajanga,Madagascar,2025-06-17 12:00:00,broken clouds,30.27,40,30.01,2.55,1012,1015


{'Temperatura (°C)': 22.872460000000025,
 'Humedad (%)': 65.00225,
 'Sensación térmica (°C)': 23.045385000000024,
 'Velocidad del viento (m/s)': 3.189285,
 'Presión atmosférica nivel del suelo (hPa)': 967.25875,
 'Presión atmosférica nivel del mar (hPa)': 1013.48025}

### b) Pronóstico de calidad del aire

In [10]:
def get_air_quality(ciudades, df_cities, api_key):
    df_air_quality = pd.DataFrame()
    for ciudad in ciudades:
        ciudad_encontrada = df_cities[df_cities["city_name"] == ciudad]
        if ciudad_encontrada.empty:
            continue
        lat = ciudad_encontrada["latitude"].values[0]
        lon = ciudad_encontrada["longitude"].values[0]
        url = f'http://api.openweathermap.org/data/2.5/air_pollution/forecast?lat={lat}&lon={lon}&appid={api_key}'
        r = requests.get(url)
        if r.status_code != 200:
            continue
        data = r.json()

        records = []
        for entry in data['list']:
            comp = entry['components']
            records.append({
                'Ciudad': ciudad,
                'datetime': pd.to_datetime(entry['dt'], unit='s'),
                'PM2.5': comp['pm2_5'],
                'PM10': comp['pm10'],
                'CO': comp['co'],
                'NO': comp['no'],
                'NO2': comp['no2'],
                'O3': comp['o3'],
                'SO2': comp['so2'],
                'NH3': comp['nh3'],
            })
        df = pd.DataFrame(records)
        df_air_quality = pd.concat([df_air_quality, df], ignore_index=True)
    promedios = df_air_quality.mean(numeric_only=True).to_dict()
    return df_air_quality, promedios

Probar la función para la ciudad de Santiago.

In [11]:
df_air_stgo, promedio_air_stgo = get_air_quality(["Santiago"], df_cities, API_KEY)
display(df_air_stgo)
display(promedio_air_stgo)

,Ciudad,datetime,PM2.5,PM10,CO,NO,NO2,O3,SO2,NH3
0,Santiago,2025-06-12 16:00:00,3.60,4.43,173.87,0.21,3.20,49.23,0.25,0.57
1,Santiago,2025-06-12 17:00:00,2.89,3.66,162.80,0.30,2.79,49.62,0.25,0.54
2,Santiago,2025-06-12 18:00:00,2.52,3.25,154.74,0.36,2.55,50.17,0.30,0.53
3,Santiago,2025-06-12 19:00:00,1.98,2.68,138.45,0.30,2.27,50.49,0.34,0.51
4,Santiago,2025-06-12 20:00:00,1.88,2.61,129.56,0.20,2.38,49.49,0.39,0.56
...,...,...,...,...,...,...,...,...,...,...
91,Santiago,2025-06-16 11:00:00,15.44,18.35,350.73,0.01,9.81,36.60,1.00,1.69
92,Santiago,2025-06-16 12:00:00,16.72,20.48,386.80,0.02,12.49,33.58,0.95,1.95
93,Santiago,2025-06-16 13:00:00,18.11,22.79,423.20,0.17,14.98,31.45,1.04,2.80
94,Santiago,2025-06-16 14:00:00,18.82,24.27,438.08,0.76,15.51,33.48,1.36,3.88


{'PM2.5': 37.3534375,
 'PM10': 42.4078125,
 'CO': 473.77104166666663,
 'NO': 0.44916666666666666,
 'NO2': 13.152083333333332,
 'O3': 39.234791666666666,
 'SO2': 2.2590625,
 'NH3': 3.949895833333334}

### c) Temperaturas extremas

Crear función para obtener tablas de wikipedia.

In [12]:
def cargar_info_wikipedia():
  url = "https://en.wikipedia.org/wiki/List_of_weather_records"
  tables = pd.read_html(url)
  df_maximas = tables[0]
  df_minimas = tables[2]
  df_maximas.columns = ['Pais', 'Temperatura máxima registrada (°C)', 'Ubicación Max', 'Fecha Max']
  df_minimas.columns = ['Pais', 'Temperatura mínima registrada (°C)', 'Ubicación Min', 'Fecha Min']

  df_extremas = pd.merge(df_maximas, df_minimas, on="Pais", how="outer")
  df_extremas['Fecha Max'] = df_extremas['Fecha Max'].astype(str).str.replace(r'\[.*?\]', '', regex=True)
  df_extremas['Fecha Min'] = df_extremas['Fecha Min'].astype(str).str.replace(r'\[.*?\]', '', regex=True)
  df_extremas.dropna(inplace=True)
  df_extremas.reset_index(drop=True, inplace=True)
  return df_extremas

Crear función para obtener temperaturas historicas extremas.

In [13]:
def get_extremas(ciudades, df_cities, df_extremas):
    df_paises = df_cities[df_cities['city_name'].isin(ciudades)][['city_name', 'country']].drop_duplicates()
    paises = df_paises['country'].tolist()
    df_historicos = df_extremas[df_extremas['Pais'].isin(paises)]
    return df_historicos

Verificar resultados de las funciones.

In [14]:
df_extremas = cargar_info_wikipedia()
df_historicos = get_extremas(["Santiago", "Buenos Aires"], df_cities, df_extremas)
df_historicos

,Pais,Temperatura máxima registrada (°C),Ubicación Max,Fecha Max,Temperatura mínima registrada (°C),Ubicación Min,Fecha Min
7,Argentina,48.9 °C (120.0 °F),"Rivadavia, Salta Province[note 7]",11 December 1905,−32.8 °C (−27.0 °F),"Sarmiento, Chubut Province",1 June 1907
20,Chile,44.9 °C (112.8 °F),"Quillón, Biobio Region[note 9]",26 January 2017,−28.5 °C (−19.3 °F),"Balmaceda, Aysen Region",16 June 1958


## Visualización interactiva con Dash

In [18]:
df_cities_json = df_cities.to_json(date_format='iso', orient='split')
df_extremas_json = df_extremas.to_json(date_format='iso', orient='split')

app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    html.H1("Consulta de Clima y Calidad del Aire"),
    dbc.Row([
        dbc.Col(dcc.Input(id='city-input', type='text', placeholder='Ingresa el nombre de una ciudad'), md=8),
        dbc.Col(dbc.Button('Consultar', id='submit-button', n_clicks=0, color="primary"), md=4),
    ], className="mb-3"),
    html.Div(id='output-message'),
    dbc.Row([
        dbc.Col(dbc.Card(id='weather-card', className="mb-3"), md=4),
        dbc.Col(dbc.Card(id='air-quality-card', className="mb-3"), md=4),
        dbc.Col(dbc.Card(id='extreme-temps-card', className="mb-3"), md=4),
    ]),
    html.Hr(id='separator-line-1'),
    html.Div(id='temp-graph-container'),
    html.Div(id='humidity-graph-container'),
    html.Hr(id='separator-line-2'),
    html.Div(id='forecast-table-container'),
    html.Div(id='air-quality-table-container'),
    html.Div(id='extreme-temps-table-container'),
    dcc.Store(id='df-cities-store', data=df_cities_json),
    dcc.Store(id='df-extremas-store', data=df_extremas_json),
    dcc.Store(id='api-key-store', data=API_KEY),
])

@app.callback(
    [
        Output('output-message', 'children'),
        Output('weather-card', 'children'),
        Output('air-quality-card', 'children'),
        Output('extreme-temps-card', 'children'),
        Output('temp-graph-container', 'children'),
        Output('humidity-graph-container', 'children'),
        Output('forecast-table-container', 'children'),
        Output('air-quality-table-container', 'children'),
        Output('extreme-temps-table-container', 'children'),
        Output('separator-line-1', 'style'),
        Output('separator-line-2', 'style'),
    ],
    [Input('submit-button', 'n_clicks')],
    [
        State('city-input', 'value'),
        State('df-cities-store', 'data'),
        State('df-extremas-store', 'data'),
        State('api-key-store', 'data'),
    ]
)
def update_output(n_clicks, city_name, df_cities_json_data, df_extremas_json_data, api_key):
    output_message = html.Div()
    weather_card_content = []
    air_quality_card_content = []
    extreme_temps_card_content = []
    temp_graph_content = html.Div()
    humidity_graph_content = html.Div()
    forecast_table_content = html.Div()
    air_quality_table_content = html.Div()
    extreme_temps_table_content = html.Div()
    separator_style = {'display': 'none'}

    if n_clicks > 0 and city_name:
        df_cities_callback = pd.read_json(df_cities_json_data, orient='split')
        df_extremas_callback = pd.read_json(df_extremas_json_data, orient='split')
        api_key_callback = api_key

        if city_name not in df_cities_callback['city_name'].values:
            output_message = dbc.Alert(f"Ciudad '{city_name}' no encontrada en nuestra base de datos.", color="warning")
        else:
            country = df_cities_callback[df_cities_callback['city_name'] == city_name]['country'].iloc[0]
            df_forecast, promedio_forecast = get_forecast([city_name], df_cities_callback, api_key_callback)
            df_air, promedio_air = get_air_quality([city_name], df_cities_callback, api_key_callback)
            df_country_extremes = df_extremas_callback[df_extremas_callback['Pais'] == country]
            weather_card_content = [
                dbc.CardHeader("Promedio Climático"),
                dbc.CardBody([
                    html.P(f"Temperatura Promedio: {promedio_forecast.get('Temperatura (°C)', 'N/A'):.2f} °C"),
                    html.P(f"Humedad Promedio: {promedio_forecast.get('Humedad (%)', 'N/A'):.2f} %"),
                    html.P(f"Velocidad del Viento Promedio: {promedio_forecast.get('Velocidad del viento (m/s)', 'N/A'):.2f} m/s"),
                ]),
            ]
            air_quality_card_content = [
                dbc.CardHeader("Promedio Calidad del Aire"),
                dbc.CardBody([
                    html.P(f"PM2.5 Promedio: {promedio_air.get('PM2.5', 'N/A'):.2f}"),
                    html.P(f"CO Promedio: {promedio_air.get('CO', 'N/A'):.2f}"),
                    html.P(f"NO2 Promedio: {promedio_air.get('NO2', 'N/A'):.2f}"),
                ]),
            ]
            extreme_temps_card_content = [
                dbc.CardHeader("Temperaturas Extremas Históricas"),
                dbc.CardBody([
                    html.P(f"Máxima Registrada: {df_country_extremes['Temperatura máxima registrada (°C)'].iloc[0] if not df_country_extremes.empty else 'N/A'} °C ({df_country_extremes['Ubicación Max'].iloc[0] if not df_country_extremes.empty else ''}, {df_country_extremes['Fecha Max'].iloc[0] if not df_country_extremes.empty else ''})"),
                    html.P(f"Mínima Registrada: {df_country_extremes['Temperatura mínima registrada (°C)'].iloc[0] if not df_country_extremes.empty else 'N/A'} °C ({df_country_extremes['Ubicación Min'].iloc[0] if not df_country_extremes.empty else ''}, {df_country_extremes['Fecha Min'].iloc[0] if not df_country_extremes.empty else ''})"),
                ]),
            ]
            if df_forecast is not None and not df_forecast.empty:
                df_forecast['Fecha y hora'] = pd.to_datetime(df_forecast['Fecha y hora'])
                df_forecast['Date'] = df_forecast['Fecha y hora'].dt.date
                daily_min_max_temp = df_forecast.groupby('Date')['Temperatura (°C)'].agg(['min', 'max']).reset_index()
                temp_graph_content = html.Div([
                    html.H3("Temperatura Mínima y Máxima por Fecha"),
                    dcc.Graph(
                        figure=px.line(
                            daily_min_max_temp,
                            x='Date',
                            y=['min', 'max'],
                            title='Temperatura Mínima y Máxima Diaria',
                            labels={'value': 'Temperatura (°C)', 'variable': 'Tipo'},
                            markers=True
                        )
                    ),
                ])
                humidity_graph_content = html.Div([
                    html.H3("Evolución de la Humedad"),
                    dcc.Graph(
                        figure=px.line(df_forecast, x='Fecha y hora', y='Humedad (%)', title='Humedad (%) a lo largo del tiempo')
                    ),
                ])
            else:
                temp_graph_content = dbc.Alert("No hay datos de pronóstico para graficar la temperatura.", color="warning")
                humidity_graph_content = dbc.Alert("No hay datos de pronóstico para graficar la humedad.", color="warning")
            if df_forecast is not None and not df_forecast.empty:
                 forecast_table_content = html.Div([
                    html.H3(f"Tabla Detallada del Pronóstico Climático para {city_name}"),
                    dbc.Table.from_dataframe(df_forecast[['Fecha y hora', 'Descripción', 'Temperatura (°C)', 'Humedad (%)', 'Sensación térmica (°C)']], striped=True, bordered=True, hover=True, responsive=True),
                ])
            if df_air is not None and not df_air.empty:
                air_quality_table_content = html.Div([
                    html.H3(f"Tabla Detallada del Pronóstico de Calidad del Aire para {city_name}"),
                     dbc.Table.from_dataframe(df_air.drop(columns=['datetime']), striped=True, bordered=True, hover=True, responsive=True),
                ])
            if not df_country_extremes.empty:
                 extreme_temps_table_content = html.Div([
                    html.H3(f"Tabla Detallada de Temperaturas Extremas Históricas para {country}"),
                    dbc.Table.from_dataframe(df_country_extremes, striped=True, bordered=True, hover=True, responsive=True),
                ])
            separator_style = {'display': 'block'}
    return (
        output_message,
        weather_card_content,
        air_quality_card_content,
        extreme_temps_card_content,
        temp_graph_content,
        humidity_graph_content,
        forecast_table_content,
        air_quality_table_content,
        extreme_temps_table_content,
        separator_style,
        separator_style,
    )


Ejecutar aplicación.

In [19]:
if __name__ == '__main__':
    app.run(debug=True, jupyter_mode="inline")

<IPython.core.display.Javascript object>